# Week 06 | Reproducible Data Pipeline
**Team 06 (TEAM-06) · Google Colab · English**

**Members:** Salem Alharbi (A) · Mohammed Almogbel (B) · Ziyad Alharbi (C) · Fahad Hamdan Alharbi (D)

**Input:** `week_06_team_project_data_S000.zip` from Google Drive. The source dataset is named **S000**; the submitting group is **Team 06**.

**Run:** In Colab, select **Runtime → Restart session and run all**.

**Workflow:** Intake → Cleaning → Integration → Analysis → Verified outputs.

## Setup — connect Google Drive
- Keep the original ZIP in **My Drive**; edit `ZIP_PATH` if it is in a folder.
- Extract the four input files before running the exercises.

In [ ]:
import io, json, hashlib, os, sqlite3, tempfile, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

STUDENT_ID = "TEAM-06"
INPUT_DIR = Path("/content/day05_team_files")
OUTPUT_DIR = Path("/content/team_output")
RAW_COLUMNS = ["request_id", "desk_code", "desk", "request_type", "status", "opened_at", "minutes"]
ACCEPTED_COLUMNS = ["source_row"] + RAW_COLUMNS
ANALYTICAL_COLUMNS = ["source", "source_row", "request_id", "desk_code", "desk", "request_type", "status", "opened_at", "minutes", "region", "staffed_hours", "month", "hours"]
REJECTION_RULES = ["conflicting_id", "unknown_desk", "bad_date", "implausible_minutes"]
MONTHS = ("jan", "feb", "mar")

### Original project data
- This notebook reads the **actual ZIP you uploaded**, not embedded or generated data.
- The ZIP must contain January and February CSVs, the March SQLite database, and `targets.csv`.

In [ ]:
from google.colab import drive
from google.colab import files

drive.mount("/content/drive")

# Locate the original data ZIP (also supports folders inside My Drive).
ZIP_NAME = "week_06_team_project_data_S000.zip"
drive_root = Path("/content/drive/MyDrive")
matches = sorted(drive_root.rglob(ZIP_NAME))
if matches:
    ZIP_PATH = matches[0]
else:
    print(f"{ZIP_NAME} was not found in Drive. Upload it from your device:")
    uploaded = files.upload()
    if ZIP_NAME not in uploaded:
        raise FileNotFoundError(f"Please select {ZIP_NAME}")
    ZIP_PATH = Path("/content") / ZIP_NAME

# Extract the required files to a fixed location, independent of Colab's working directory.
expected = {f"day05_team_files/{name}" for name in
            ("jan_requests.csv", "feb_requests.csv", "team_data.db", "targets.csv")}
with zipfile.ZipFile(ZIP_PATH) as archive:
    archived = set(archive.namelist())
    if archived != expected:
        raise ValueError(f"Unexpected ZIP contents: {sorted(archived)}")
    archive.extractall("/content")

missing = [name for name in sorted(expected) if not (Path("/content") / name).is_file()]
if missing:
    raise FileNotFoundError(f"These files were not extracted: {missing}")
print("Data ZIP:", ZIP_PATH)
print("Input folder:", INPUT_DIR)
print("Files:", sorted(p.name for p in INPUT_DIR.iterdir()))


## Team plan / interface contracts
- Each role returns a DataFrame with the agreed column types; invalid interfaces raise `ValueError`.
- Each member owns a distinct function and one test; integration happens after the role checks pass.

In [ ]:
TEAM_PLAN = {
    "A": {"owner": "Salem Alharbi", "tasks": ["EX01", "EX02"], "handoff": "raw_month"},
    "B": {"owner": "Mohammed Almogbel", "tasks": ["EX03", "EX04"], "handoff": "accepted"},
    "C": {"owner": "Ziyad Alharbi", "tasks": ["EX05", "EX06"], "handoff": "analytical"},
    "D": {"owner": "Fahad Hamdan Alharbi", "tasks": ["EX07", "EX08", "EX10"], "handoff": "figures and brief"},
    "TEAM": {"tasks": ["EX09"], "handoff": "verified outputs and manifest"},
}
CONTRIBUTIONS = {"A": ["load_sources", "intake_report"],
                 "B": ["clean_month", "clean_all_months"],
                 "C": ["integrate", "compare_targets"],
                 "D": ["summaries", "make_team_figures", "brief_facts"],
                 "TEAM": ["run_pipeline"]}

def check_interface(df, name):
    """Enforce text, number and date contracts at all three hand-off points."""
    required = {"raw_month": RAW_COLUMNS, "accepted": ACCEPTED_COLUMNS,
                "analytical": ANALYTICAL_COLUMNS}
    if name not in required or not isinstance(df, pd.DataFrame):
        raise ValueError(f"Unknown or non-DataFrame interface: {name}")
    if not df.columns.is_unique:
        raise ValueError(f"{name}: duplicate column names")
    missing = [c for c in required[name] if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: missing columns {missing}")

    text_fields = RAW_COLUMNS if name == "raw_month" else [
        "request_id", "desk_code", "desk", "request_type", "status"]
    if name == "analytical":
        text_fields = ["source", *text_fields, "region"]
    for field in text_fields:
        series = df[field]
        if not (pd.api.types.is_string_dtype(series.dtype) or
                pd.api.types.is_object_dtype(series.dtype)):
            raise ValueError(f"{name}: {field} must be text")
        if not series.dropna().map(lambda value: isinstance(value, str)).all():
            raise ValueError(f"{name}: {field} contains non-text values")

    if name != "raw_month":
        if not pd.api.types.is_integer_dtype(df["source_row"]):
            raise ValueError(f"{name}: source_row must be an integer")
        if not pd.api.types.is_datetime64_any_dtype(df["opened_at"]):
            raise ValueError(f"{name}: opened_at must be datetime")
        if not pd.api.types.is_numeric_dtype(df["minutes"]):
            raise ValueError(f"{name}: minutes must be numeric")
    if name == "analytical":
        if not pd.api.types.is_numeric_dtype(df["hours"]):
            raise ValueError("analytical: hours must be numeric")
        if not pd.api.types.is_integer_dtype(df["month"]):
            raise ValueError("analytical: month must be an integer")
        if not pd.api.types.is_numeric_dtype(df["staffed_hours"]):
            raise ValueError("analytical: staffed_hours must be numeric")
    return True

## EX01 — `load_sources` (Role A · 10 points)
- Read CSV fields as text; query SQLite **read-only** with explicit columns.
- Convert targets to integers and preserve February's extra `channel` column.

In [ ]:
def load_sources(paths):
    p = {k: Path(v) for k, v in paths.items()}
    jan = pd.read_csv(p["jan"], dtype=str, keep_default_na=False)
    feb = pd.read_csv(p["feb"], dtype=str, keep_default_na=False)
    db_uri = f"file:{p['db'].resolve().as_posix()}?mode=ro"
    with sqlite3.connect(db_uri, uri=True) as con:
        mar = pd.read_sql_query(
            "SELECT request_id, desk_code, desk, request_type, status, opened_at, minutes FROM march_requests", con
        ).fillna("").astype(str)
        desks = pd.read_sql_query(
            "SELECT desk_code, region, staffed_hours FROM desks", con,
            dtype={"desk_code": str, "region": str, "staffed_hours": int}
        )
    targets = pd.read_csv(p["targets"], dtype={"region": str, "month": int, "target_requests": int})
    for table in (jan, feb, mar):
        check_interface(table, "raw_month")
    if not {"desk_code", "region", "staffed_hours"}.issubset(desks.columns):
        raise ValueError("desks: missing required columns")
    if not {"region", "month", "target_requests"}.issubset(targets.columns):
        raise ValueError("targets: missing required columns")
    return {"jan": jan, "feb": feb, "mar": mar, "desks": desks, "targets": targets}

PATHS = {"jan": INPUT_DIR / "jan_requests.csv", "feb": INPUT_DIR / "feb_requests.csv",
         "db": INPUT_DIR / "team_data.db", "targets": INPUT_DIR / "targets.csv"}
sources = load_sources(PATHS)
print("Raw rows:", {m: len(sources[m]) for m in MONTHS})

## EX02 — `intake_report` (Role A · 8 points)
- Report raw row counts, exact duplicates, missing-value tokens and extra columns.
- Check the raw data **before cleaning** so no issue is hidden.

In [ ]:
def intake_report(sources):
    rows = []
    tokens = {"", "unknown", "n/a"}
    for month in MONTHS:
        df = sources[month]
        vals = df[RAW_COLUMNS].apply(lambda col: col.astype(str).str.strip().str.lower().isin(tokens))
        rows.append({"source": month, "raw_rows": int(len(df)),
                     "exact_duplicates": int(df.duplicated().sum()),
                     "missing_tokens": int(vals.to_numpy().sum()),
                     "extra_columns": ",".join(c for c in df.columns if c not in RAW_COLUMNS)})
    return pd.DataFrame(rows)

intake = intake_report(sources)
display(intake)

## EX03 — `clean_month` (Role B · 12 points)
- Apply the course policy: exact duplicates first; normalize text/codes, parse minutes/dates and flag failures.
- Reject **all conflicting IDs**, unknown desks, invalid dates and out-of-range minutes; **keep missing minutes**.

In [ ]:
def clean_month(raw_month, valid_codes, max_minutes=480):
    check_interface(raw_month, "raw_month")
    raw = raw_month.copy(deep=True).reset_index(drop=True)
    raw.insert(0, "source_row", np.arange(1, len(raw) + 1))
    # Compare ALL original raw fields, including February's channel, before dropping extras.
    original_cols = [c for c in raw.columns if c != "source_row"]
    exact = raw.duplicated(subset=original_cols, keep="first")
    work = raw.loc[~exact, ACCEPTED_COLUMNS].copy()
    work["desk_code"] = work["desk_code"].str.strip().apply(lambda x: x.zfill(3) if x.isdigit() else x)
    work["desk"] = work["desk"].str.strip().str.title()
    work["request_type"] = work["request_type"].str.strip().str.title()
    work["status"] = work["status"].str.strip().str.lower()
    work["request_id"] = work["request_id"].str.strip()
    raw_minutes = work["minutes"].str.strip()
    suffix = raw_minutes.str.extract(r"^(\d+(?:\.\d+)?)\s*min$")[0]
    work["minutes"] = pd.to_numeric(raw_minutes.where(suffix.isna(), suffix), errors="coerce")
    work["opened_at"] = pd.to_datetime(work["opened_at"], format="%Y-%m-%d %H:%M", errors="coerce")
    flags = pd.DataFrame(index=work.index)
    flags["conflicting_id"] = work.duplicated("request_id", keep=False)
    flags["unknown_desk"] = ~work["desk_code"].isin(set(valid_codes))
    flags["bad_date"] = work["opened_at"].isna()
    flags["implausible_minutes"] = work["minutes"].notna() & ((work["minutes"] <= 0) | (work["minutes"] > max_minutes))
    bad = flags.any(axis=1)
    reason_text = flags.apply(lambda row: ";".join([n for n in REJECTION_RULES if bool(row[n])]), axis=1)
    accepted = work.loc[~bad, ACCEPTED_COLUMNS].reset_index(drop=True)
    rejected = work.loc[bad, ACCEPTED_COLUMNS].copy()
    rejected["reasons"] = reason_text.loc[bad]
    rejected = rejected.reset_index(drop=True)
    report = {"raw_rows": int(len(raw)), "exact_duplicates": int(exact.sum()),
              "accepted": int(len(accepted)), "rejected": int(len(rejected))}
    assert report["raw_rows"] == report["exact_duplicates"] + report["accepted"] + report["rejected"]
    check_interface(accepted, "accepted")
    return accepted, rejected, report

valid_codes = set(sources["desks"]["desk_code"].astype(str).str.strip())
accepted_jan, rejected_jan, report_jan = clean_month(sources["jan"], valid_codes)
print("January:", report_jan)

## EX04 — Cleaning reconciliation (Role B · 8 points)
- Run the **same cleaner** for all three sources and keep each accepted/rejected table.
- Prove `raw = exact duplicates + accepted + rejected` for every month.

In [ ]:
def clean_all_months(sources, valid_codes):
    cleaned, summary = {}, []
    for month in MONTHS:
        accepted, rejected, report = clean_month(sources[month], valid_codes)
        cleaned[month] = {"accepted": accepted, "rejected": rejected, "report": report}
        summary.append({"source": month, **report,
                        "reconciled": report["raw_rows"] == report["exact_duplicates"] + report["accepted"] + report["rejected"]})
    return cleaned, pd.DataFrame(summary)

cleaned, cleaning_summary = clean_all_months(sources, valid_codes)
display(cleaning_summary)
assert cleaning_summary["reconciled"].all()

## EX05 — `integrate` (Role C · 10 points)
- Concatenate accepted months, add `source`, then enrich using **validated many-to-one** desk lookup.
- Create `month` and `hours`, and return the agreed column order.

In [ ]:
def integrate(cleaned, desks):
    lookup = desks[["desk_code", "region", "staffed_hours"]].copy()
    lookup["desk_code"] = lookup["desk_code"].astype(str).str.strip()
    lookup["desk_code"] = lookup["desk_code"].apply(lambda x: x.zfill(3) if x.isdigit() else x)
    if lookup["desk_code"].duplicated().any():
        raise ValueError("Duplicate desk_code in lookup")
    if lookup[["desk_code", "region", "staffed_hours"]].isna().any().any():
        raise ValueError("Missing lookup values")
    frames = []
    for month in MONTHS:
        accepted = cleaned[month]["accepted"]
        check_interface(accepted, "accepted")
        frames.append(accepted.assign(source=month))
    combined = pd.concat(frames, ignore_index=True)
    if combined["request_id"].duplicated().any():
        raise ValueError("Accepted request_id duplicated across months")
    analytical = combined.merge(lookup, on="desk_code", how="left", validate="many_to_one", indicator=True)
    if not analytical["_merge"].eq("both").all():
        raise ValueError("Accepted records have unmatched desk codes")
    analytical = analytical.drop(columns="_merge")
    analytical["staffed_hours"] = pd.to_numeric(analytical["staffed_hours"], errors="raise")
    analytical["month"] = analytical["opened_at"].dt.month.astype(int)
    analytical["hours"] = analytical["minutes"] / 60
    analytical = analytical[ANALYTICAL_COLUMNS].sort_values(["request_id", "source_row"], kind="stable").reset_index(drop=True)
    check_interface(analytical, "analytical")
    return analytical

analytical = integrate(cleaned, sources["desks"])
print("Analytical records:", len(analytical)); display(analytical.head(3))

## EX06 — `compare_targets` (Role C · 10 points)
- Start from **all targets**, even where zero requests were accepted; validate one-to-one merge.
- Compute actual requests and `gap = actual_requests - target_requests`.

In [ ]:
def compare_targets(analytical, targets):
    check_interface(analytical, "analytical")
    goals = targets[["region", "month", "target_requests"]].copy()
    goals["month"] = goals["month"].astype(int)
    goals["target_requests"] = goals["target_requests"].astype(int)
    if goals.duplicated(["region", "month"]).any():
        raise ValueError("Duplicate region/month targets")
    actual = analytical.groupby(["region", "month"], as_index=False).size().rename(columns={"size": "actual_requests"})
    result = goals.merge(actual, how="left", on=["region", "month"], validate="one_to_one")
    result["actual_requests"] = result["actual_requests"].fillna(0).astype(int)
    result["gap"] = result["actual_requests"] - result["target_requests"]
    return result.sort_values(["region", "month"], kind="stable").reset_index(drop=True)

vs_target = compare_targets(analytical, sources["targets"])
display(vs_target)

## EX07 — `summaries` (Role D · 10 points)
- Produce a region-by-month pivot of **accepted request counts**.
- Summarize staffed desk volume and handling minutes without inventing missing values.

In [ ]:
def summaries(analytical):
    check_interface(analytical, "analytical")
    region_month = analytical.pivot_table(index="region", columns="month", values="request_id", aggfunc="count", fill_value=0)
    region_month = region_month.reindex(columns=[1, 2, 3], fill_value=0).astype(int).reset_index()
    region_month.columns = ["region", "month_1", "month_2", "month_3"]
    desks_summary = (analytical.groupby(["desk_code", "desk", "region"], as_index=False)
                     .agg(requests=("request_id", "size"), total_minutes=("minutes", "sum"),
                          mean_minutes=("minutes", "mean")))
    desks_summary = desks_summary.sort_values("desk_code", kind="stable").reset_index(drop=True)
    return region_month.sort_values("region").reset_index(drop=True), desks_summary

region_month, desks_summary = summaries(analytical)
display(region_month); display(desks_summary)

## EX08 — `make_team_figures` (Role D · 10 points)
- Compare **January, February and March** in all three charts: requests by region, handling minutes by desk, and actual vs target.
- Save three clear PNG figures with labels, legends and consistent month colors.

In [ ]:
def make_team_figures(analytical, desks_summary, vs_target):
    month_names = {1: "Jan", 2: "Feb", 3: "Mar"}
    month_colors = {1: "#4F63D9", 2: "#20B9B0", 3: "#F4AD53"}
    figures = {}

    def save(name, fig, auto_layout=True):
        if auto_layout:
            fig.tight_layout(pad=1.6)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=150, facecolor="white",
                    metadata={"Software": "Matplotlib"})
        plt.close(fig)
        figures[name] = buf.getvalue()

    # 1. Accepted requests: regions side by side, three months per region.
    region_month = (analytical.groupby(["region", "month"]).size()
                    .unstack(fill_value=0).reindex(columns=[1, 2, 3], fill_value=0))
    regions = list(region_month.index)
    x = np.arange(len(regions))
    width = 0.24
    fig, ax = plt.subplots(figsize=(10, 5.0))
    for j, m in enumerate((1, 2, 3)):
        bars = ax.bar(x + (j - 1) * width, region_month[m], width,
                      color=month_colors[m], label=month_names[m])
        ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_xticks(x, regions)
    ax.set(title="Accepted requests by region and month", xlabel="Region", ylabel="Accepted requests")
    ax.legend(title="Month", ncol=3)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    save("requests_by_region.png", fig)

    # 2. Handling minutes: stacked monthly contribution for each desk.
    desk_month = (analytical.pivot_table(index="desk_code", columns="month",
                  values="minutes", aggfunc="sum", fill_value=0)
                  .reindex(columns=[1, 2, 3], fill_value=0))
    desk_month = desk_month.sort_index()
    x = np.arange(len(desk_month))
    bottom = np.zeros(len(desk_month))
    fig, ax = plt.subplots(figsize=(10, 5.0))
    for m in (1, 2, 3):
        values = desk_month[m].to_numpy(dtype=float)
        ax.bar(x, values, bottom=bottom, color=month_colors[m],
               label=month_names[m], width=0.66)
        bottom += values
    ax.set_xticks(x, list(desk_month.index))
    ax.set(title="Total handling minutes by desk and month",
           xlabel="Desk code", ylabel="Handling minutes (sum)")
    ax.legend(title="Month", ncol=3)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    save("minutes_by_desk.png", fig)

    # 3. Actual vs target: one panel for each month, grouped by region.
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.8), sharey=True)
    for ax, m in zip(axes, (1, 2, 3)):
        d = vs_target.loc[vs_target["month"] == m].sort_values("region")
        xx = np.arange(len(d))
        bars_actual = ax.bar(xx - 0.18, d["actual_requests"], 0.36,
                             color="#4F63D9", label="Actual")
        bars_target = ax.bar(xx + 0.18, d["target_requests"], 0.36,
                             color="#9DAABD", label="Target")
        ax.bar_label(bars_actual, padding=2, fontsize=8)
        ax.bar_label(bars_target, padding=2, fontsize=8)
        ax.set_xticks(xx, d["region"])
        ax.set(title=month_names[m], xlabel="Region")
        ax.grid(axis="y", alpha=0.18)
        ax.set_axisbelow(True)
    axes[0].set_ylabel("Accepted requests")
    fig.suptitle("Actual requests vs targets | by region and month", fontsize=14, y=0.98)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2,
               bbox_to_anchor=(0.5, 0.91), frameon=False)
    fig.subplots_adjust(left=0.07, right=0.985, bottom=0.13, top=0.75, wspace=0.12)
    save("gap_vs_target.png", fig, auto_layout=False)  # Preserve output filename.
    return figures

figures = make_team_figures(analytical, desks_summary, vs_target)
print("Charts:", list(figures))

## EX09 — `run_pipeline` (Team · 12 points)
- **Validate first**, then generate four CSVs, three PNGs and a manifest in stable order.
- Verify byte-identical reruns; an invalid desk lookup must raise `ValueError` **before writing**.

In [ ]:
def _csv_bytes(df):
    return df.to_csv(index=False, lineterminator="\n", date_format="%Y-%m-%d %H:%M", float_format="%.6f", na_rep="").encode("utf-8")

def run_pipeline(paths, out_dir, valid_codes=None):
    data = load_sources(paths)
    lookup_codes = set(data["desks"]["desk_code"].astype(str).str.strip())
    lookup_codes = {c.zfill(3) if c.isdigit() else c for c in lookup_codes}
    if valid_codes is not None and {str(c).strip().zfill(3) if str(c).strip().isdigit() else str(c).strip() for c in valid_codes} != lookup_codes:
        raise ValueError("valid_codes does not match the authoritative desks lookup")
    # Critical preflight, before output directory creation.
    raw_lookup = data["desks"]["desk_code"].astype(str).str.strip()
    normalized_lookup = raw_lookup.apply(lambda c: c.zfill(3) if c.isdigit() else c)
    if normalized_lookup.duplicated().any():
        raise ValueError("Duplicate desk_code in SQLite lookup")
    cleaned, summary = clean_all_months(data, lookup_codes)
    table = integrate(cleaned, data["desks"])
    comparison = compare_targets(table, data["targets"])
    by_region, by_desk = summaries(table)
    images = make_team_figures(table, by_desk, comparison)
    files = {"analytical_table.csv": _csv_bytes(table),
             "region_month.csv": _csv_bytes(by_region),
             "vs_target.csv": _csv_bytes(comparison),
             "cleaning_summary.csv": _csv_bytes(summary), **images}
    manifest = {
        "student_id": STUDENT_ID,
        "raw_rows": int(summary["raw_rows"].sum()),
        "exact_duplicates": int(summary["exact_duplicates"].sum()),
        "accepted": int(summary["accepted"].sum()),
        "rejected": int(summary["rejected"].sum()),
        "reconciled": bool(summary["reconciled"].all()),
        "source_rows": {r["source"]: int(r["raw_rows"]) for r in summary.to_dict("records")},
        "files_sha256": {k: hashlib.sha256(v).hexdigest() for k, v in sorted(files.items())},
    }
    files["manifest.json"] = (json.dumps(manifest, indent=2, sort_keys=True) + "\n").encode("utf-8")
    # All validation is complete; write only now.
    destination = Path(out_dir)
    destination.mkdir(parents=True, exist_ok=True)
    for name in sorted(files):
        (destination / name).write_bytes(files[name])
    return {"manifest": manifest, "files": files, "cleaned": cleaned,
            "analytical": table, "vs_target": comparison, "region_month": by_region,
            "cleaning_summary": summary, "desks_summary": by_desk}

result = run_pipeline(PATHS, OUTPUT_DIR)
print(json.dumps(result["manifest"], indent=2))

### Mandatory verification
- Re-run the pipeline and compare every output byte; reconcile all source rows.
- Deliberately duplicate a SQLite desk in a separate copy: the pipeline must fail without creating output files.

In [ ]:
import shutil
first_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in OUTPUT_DIR.iterdir() if p.is_file()}
rerun = run_pipeline(PATHS, OUTPUT_DIR)
second_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in OUTPUT_DIR.iterdir() if p.is_file()}
assert first_hashes == second_hashes, "Rerun changed output bytes"
assert result["manifest"]["reconciled"]
assert result["manifest"]["raw_rows"] == result["manifest"]["exact_duplicates"] + result["manifest"]["accepted"] + result["manifest"]["rejected"]
assert len(first_hashes) == 8
# Confirm the missing-minutes rule on an explicit valid request.
missing_sample = pd.DataFrame([{ "request_id": "TEST-MISSING", "desk_code": "001",
    "desk": "North", "request_type": "Report", "status": "open",
    "opened_at": "2025-01-01 10:00", "minutes": "unknown"}])
miss_acc, miss_rej, _ = clean_month(missing_sample, set(sources["desks"]["desk_code"].astype(str)))
assert len(miss_acc) == 1 and len(miss_rej) == 0 and pd.isna(miss_acc.loc[0, "minutes"])

# Contract regression checks: incorrect types must raise ValueError.
for field in ("desk_code", "request_id"):
    broken = cleaned["jan"]["accepted"].copy()
    broken[field] = 7
    try:
        check_interface(broken, "accepted")
        raise AssertionError(f"Incorrect {field} type was accepted")
    except ValueError:
        pass
broken = result["analytical"].copy()
broken["staffed_hours"] = "not numeric"
try:
    check_interface(broken, "analytical")
    raise AssertionError("Incorrect staffed_hours type was accepted")
except ValueError:
    pass
with tempfile.TemporaryDirectory() as temp:
    db_copy = Path(temp) / "broken.db"
    shutil.copy2(PATHS["db"], db_copy)
    with sqlite3.connect(db_copy) as conn:
        conn.execute("INSERT INTO desks SELECT * FROM desks LIMIT 1")
        conn.commit()
    bad_paths = dict(PATHS, db=db_copy)
    bad_out = Path(temp) / "should_not_exist"
    try:
        run_pipeline(bad_paths, bad_out)
        raise AssertionError("Duplicate desk lookup was accepted")
    except ValueError as e:
        assert "Duplicate desk_code" in str(e)
    assert not bad_out.exists(), "Output created despite failed preflight"
print("PASS: 8 outputs; byte-identical rerun; reconciliation; type contracts; missing durations; safe failure")

## EX10 — `brief_facts` + contributions (Role D + Team · 10 points)
- Generate facts **from actual outputs**, then provide a 100–180-word brief with two limitations.
- Record each role's contribution and a concise 100–180-word findings brief.

In [ ]:
def brief_facts(result):
    summary = result["cleaning_summary"]
    table = result["analytical"]
    vs = result["vs_target"]
    totals = summary[["raw_rows", "exact_duplicates", "accepted", "rejected"]].sum()
    by_region = table.groupby("region").size().sort_values(ascending=False)
    largest = by_region.index[0] if len(by_region) else "none"
    missing_minutes = int(table["minutes"].isna().sum())
    above = int((vs["gap"] > 0).sum())
    below = int((vs["gap"] < 0).sum())
    facts = {"raw_rows": int(totals["raw_rows"]), "exact_duplicates": int(totals["exact_duplicates"]),
             "accepted": int(totals["accepted"]), "rejected": int(totals["rejected"]),
             "top_region": largest, "top_region_requests": int(by_region.iloc[0]) if len(by_region) else 0,
             "missing_minutes_accepted": missing_minutes, "above_target_pairs": above, "below_target_pairs": below}
    brief = (
        f"This reproducible service-request pipeline processed {facts['raw_rows']} raw records across January, February and March. "
        f"The cleaning policy removed {facts['exact_duplicates']} exact duplicate rows, accepted {facts['accepted']} records, "
        f"and rejected {facts['rejected']} records under documented quality rules. All monthly counts reconcile. "
        f"The {facts['top_region']} region had the largest accepted volume, with {facts['top_region_requests']} requests. "
        f"Among the nine region-month comparisons, {facts['above_target_pairs']} exceeded their targets and "
        f"{facts['below_target_pairs']} fell below their targets. {facts['missing_minutes_accepted']} accepted requests "
        "have missing handling durations; these values were retained as missing rather than guessed or silently discarded. "
        "The output includes a validated analytical table, a regional pivot, a targets comparison, a cleaning summary and three charts. "
        "A manifest records the result totals and file checksums, allowing byte-for-byte checks after a repeated run. "
        "Two limitations matter: the data is synthetic, so it does not establish real police service performance, "
        "and the course-defined handling-time ceiling can misclassify legitimate unusually long cases. "
        "We recommend reviewing conflicting request identifiers and invalid timestamps with the data owner before using these results for decisions."
    )
    assert 100 <= len(brief.split()) <= 180, len(brief.split())
    return facts, brief

facts, BRIEF = brief_facts(result)
print("Facts:", facts)
print("\nBrief (", len(BRIEF.split()), "words):\n", BRIEF)

# Local contribution evidence only. The original course notebook's auto-grader
# is required to create or verify the official skill-score report.
ROLE_SKILLS = {"A": ["S1"], "B": ["S2"], "C": ["S3"], "D": ["S4", "S6"]}
SKILL_REPORT = {"team": STUDENT_ID,
    "shared_skill": "S5",
    "official_autograder_verified": False,
    "contributions": [
        {"role": role, "owner": TEAM_PLAN[role]["owner"],
         "skills": ROLE_SKILLS[role], "functions": funcs,
         "status": "Local checks passed; official auto-grade not available"}
        for role, funcs in CONTRIBUTIONS.items() if role != "TEAM"
    ]}
Path(f"results_W06-D5-B_{STUDENT_ID}.json").write_text(json.dumps(SKILL_REPORT, indent=2) + "\n", encoding="utf-8")
assert all(callable(globals()[fn]) for names in CONTRIBUTIONS.values() for fn in names)

## Bonus Analysis — SQL-Based Decision Insights (Mohammed Almogbel)
- These **optional analyses run after EX01–EX10** on the accepted analytical table. The required **4 CSV + 3 PNG + manifest** remain unchanged.
- SQL results and two additional figures are saved separately in `bonus_output/`; the findings are exploratory and the dataset is synthetic.

In [ ]:
# Bonus A/B — SQL queries on the validated analytical table; original SQLite is untouched.
BONUS_DIR = Path("/content/bonus_output")
BONUS_DIR.mkdir(parents=True, exist_ok=True)

bonus_conn = sqlite3.connect(":memory:")
result["analytical"].to_sql("analytical", bonus_conn, index=False, if_exists="replace")

handling_sql = """
SELECT request_type, month,
       COUNT(*) AS accepted_requests,
       COUNT(minutes) AS known_duration_count,
       ROUND(AVG(minutes), 2) AS avg_minutes
FROM analytical
GROUP BY request_type, month
ORDER BY request_type, month
"""
quality_sql = """
SELECT region, month,
       COUNT(*) AS accepted_requests,
       SUM(CASE WHEN minutes IS NULL THEN 1 ELSE 0 END) AS missing_minutes,
       ROUND(100.0 * SUM(CASE WHEN minutes IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_pct
FROM analytical
GROUP BY region, month
ORDER BY region, month
"""

handling_sql_result = pd.read_sql_query(handling_sql, bonus_conn)
quality_sql_result = pd.read_sql_query(quality_sql, bonus_conn)
bonus_conn.close()

assert handling_sql_result["accepted_requests"].sum() == len(result["analytical"])
assert quality_sql_result["accepted_requests"].sum() == len(result["analytical"])
assert quality_sql_result["missing_minutes"].sum() == result["analytical"]["minutes"].isna().sum() == 36
assert handling_sql_result["known_duration_count"].sum() == result["analytical"]["minutes"].notna().sum()
handling_sql_result.to_csv(BONUS_DIR / "handling_by_type_month.csv", index=False)
quality_sql_result.to_csv(BONUS_DIR / "missing_minutes_by_region_month.csv", index=False)
print("Bonus A — request handling durations:")
display(handling_sql_result)
print("Bonus B — duration completeness:")
display(quality_sql_result)


In [ ]:
# Two optional charts. No change to the three mandatory EX08 charts.
month_labels = {1: "Jan", 2: "Feb", 3: "Mar"}
series_colors = {"Traffic": "#4B86CA", "Lost Item": "#0BAA9D", "Report": "#9CA9BD", "Document": "#DE9D58"}
fig, ax = plt.subplots(figsize=(10.8, 5.0))
for category in ["Traffic", "Lost Item", "Report", "Document"]:
    rows = handling_sql_result[handling_sql_result["request_type"] == category].sort_values("month")
    ax.plot(rows["month"], rows["avg_minutes"], marker="o", markersize=8,
            linewidth=2.8 if category in ("Traffic", "Lost Item") else 1.8,
            color=series_colors[category], label=category)
    if category in ("Traffic", "Lost Item"):
        feb_point = rows.loc[rows["month"] == 2, "avg_minutes"]
        if not feb_point.empty:
            ax.annotate(f"{feb_point.iloc[0]:.1f}", (2, feb_point.iloc[0]),
                        xytext=(0, 9), textcoords="offset points",
                        fontsize=10, ha="center", weight="bold")
ax.set_xticks([1, 2, 3], ["January", "February", "March"])
ax.set_ylim(17, 37.5)
ax.set_ylabel("Mean recorded handling minutes (not total requests)")
ax.set_title("Bonus A | Which request types changed in February?", weight="bold", loc="left", pad=16)
ax.grid(axis="y", alpha=.19)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(.5, -0.24), frameon=False)
fig.subplots_adjust(left=.10, right=.97, top=.85, bottom=.25)
fig.savefig(BONUS_DIR / "handling_time_by_request_type.png", dpi=170, facecolor="white",
            metadata={"Software": "Matplotlib"})
plt.show(); plt.close(fig)

regions = sorted(quality_sql_result["region"].unique())
heat = (quality_sql_result.pivot(index="region", columns="month", values="missing_pct")
        .reindex(index=regions, columns=[1, 2, 3]))
missing_ct = quality_sql_result.set_index(["region", "month"])
fig, ax = plt.subplots(figsize=(10, 4.6))
from matplotlib.colors import LinearSegmentedColormap
palette = LinearSegmentedColormap.from_list("quality", ["#D8F1E9", "#FCE7B0", "#E9A7A1"])
im = ax.imshow(heat.to_numpy(), cmap=palette, vmin=0, vmax=12, aspect="auto")
ax.set_xticks([0, 1, 2], ["January", "February", "March"])
ax.set_yticks(range(len(regions)), regions)
for i, region in enumerate(regions):
    for j, month in enumerate([1, 2, 3]):
        d = missing_ct.loc[(region, month)]
        ax.text(j, i, f"{d['missing_pct']:.1f}%\n{int(d['missing_minutes'])}/{int(d['accepted_requests'])} missing",
                ha="center", va="center", fontsize=12, weight="bold", color="#23344B")
ax.set_title("Bonus B | Missing handling minutes by region and month", weight="bold", loc="left", pad=16)
for sp in ax.spines.values(): sp.set_visible(False)
fig.subplots_adjust(left=.17, right=.96, top=.84, bottom=.17)
fig.savefig(BONUS_DIR / "missing_minutes_heatmap.png", dpi=170, facecolor="white",
            metadata={"Software": "Matplotlib"})
plt.show(); plt.close(fig)

jan = handling_sql_result.query("month == 1").set_index("request_type")
feb = handling_sql_result.query("month == 2").set_index("request_type")
traffic_change = 100 * (feb.loc["Traffic", "avg_minutes"] / jan.loc["Traffic", "avg_minutes"] - 1)
lost_change = 100 * (feb.loc["Lost Item", "avg_minutes"] / jan.loc["Lost Item", "avg_minutes"] - 1)
metro_missing = int(quality_sql_result.query("region == 'Metro'")["missing_minutes"].sum())
print(f"Traffic: {traffic_change:+.1f}% February vs January; Lost Item: {lost_change:+.1f}%")
print(f"Missing durations: 36 of 521 (Metro: {metro_missing}). These are signals to review, not proof of causes.")


## Final submission
- Keep `week_06_team_project_data_S000.zip` in Google Drive; run every cell from the beginning.
- Submit this notebook, the `team_output` directory (**4 CSV, 3 PNG (monthly comparisons), 1 manifest**) and the Team 06 skill-report JSON.
- The last cell also saves `Week06_Team06_Results.zip` in Google Drive.
- Hidden grading still requires the original Day-5 course notebook and its interface checks.
- `bonus_output/` contains two **optional** SQL result CSVs and charts for Mohammed’s extra insights; these do not replace official outputs.

In [ ]:
print("Deliverables:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)
report_path = Path(f"results_W06-D5-B_{STUDENT_ID}.json")
print(" -", report_path.name)

results_zip = Path("/content/drive/MyDrive/Week06_Team06_Results.zip")
with zipfile.ZipFile(results_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(OUTPUT_DIR.iterdir()):
        archive.write(p, arcname=f"team_output/{p.name}")
    archive.write(report_path, arcname=report_path.name)
    for p in sorted(BONUS_DIR.iterdir()):
        archive.write(p, arcname=f"bonus_output/{p.name}")
print("Saved to Google Drive:", results_zip)
print("Extra SQL charts:", sorted(p.name for p in BONUS_DIR.iterdir()))
from IPython.display import Image
for chart_name in ("requests_by_region.png", "minutes_by_desk.png", "gap_vs_target.png"):
    print("\n" + chart_name)
    display(Image(filename=str(OUTPUT_DIR / chart_name)))

## AI Project — EDA and data preparation for AI (Team 06)
- **Goal:** take a deliberately messy dataset and prepare it for a machine-learning model, measuring every decision instead of assuming it. This is EDA for modelling, not for a report.
- **Data:** a synthetic service-request dataset generated here with planted problems: missing values that follow patterns, high dispersion with extreme values, skewed distributions, and one region that is badly under-represented (sampling bias). Its request types, regions and typical durations are learned from the pipeline's own analytical table.
- **Steps:** AI-1 generate → AI-2 explore → AI-3 study and fill the gaps → AI-4 treat dispersion and skew → AI-5 balance with synthetic data (SMOTE-NC) → AI-6 unsupervised model, compared before and after preparation.
- **Scope:** runs after EX01–EX10 and the SQL bonus and writes only to `ai_output/`. The required **4 CSV + 3 PNG + manifest** are unchanged. All randomness uses fixed seeds.

### AI-1 — Generate a messy dataset with known problems
- Because the data is generated, the complete "truth" is kept aside. It is **never** used to make decisions, only to check at the end how good the filled-in values were.
- Planted problems: gaps that depend on request type, region and waiting time; right-skewed durations with extreme values; a 9 : 3 : 1 imbalance between regions.

In [ ]:
from imblearn.over_sampling import SMOTENC
from scipy.stats import ks_2samp
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import KNNImputer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import PowerTransformer, StandardScaler

AI_DIR = Path("/content/ai_output")
AI_SEED = 6                                   # fixed seed: Team 06
GROUP = "region"                              # the group whose representation is biased
NUMERIC = ["handling_minutes", "wait_minutes", "staff_on_duty", "satisfaction"]
CATEGORICAL = ["request_type", "channel"]
MODEL_NUMERIC = NUMERIC + ["hour"]
SKEW_LIMIT = 1.0                              # |skew| above this counts as strongly skewed
OUTLIER_LIMIT = 0.05                          # more than 5% IQR outliers counts as heavy tails

def make_messy_dataset(analytical, n_rows=1500, seed=AI_SEED):
    """Synthetic requests with planted problems. Returns (truth, messy): same rows, messy has the gaps."""
    rng = np.random.default_rng(seed)
    regions = sorted(analytical["region"].dropna().unique())
    types = sorted(analytical["request_type"].dropna().unique())
    typical = analytical.groupby("request_type")["minutes"].median().to_dict()   # learned from real pipeline data
    region_weights = np.array([9.0, 3.0, 1.0, 0.6, 0.4, 0.3][:len(regions)])     # sampling bias between regions
    type_weights = np.array([4.0, 3.0, 2.0, 1.0, 0.8, 0.6][:len(types)])
    hours = np.arange(7, 20)
    hour_weights = np.array([2, 4, 7, 10, 11, 9, 6, 5, 6, 5, 4, 3, 2], dtype=float)

    region = rng.choice(regions, size=n_rows, p=region_weights / region_weights.sum())
    request_type = rng.choice(types, size=n_rows, p=type_weights / type_weights.sum())
    channel = rng.choice(["counter", "phone", "online"], size=n_rows, p=[0.55, 0.30, 0.15])
    hour = rng.choice(hours, size=n_rows, p=hour_weights / hour_weights.sum())
    region_rank = np.array([regions.index(r) for r in region])
    staff = np.clip(rng.poisson(6 - 1.3 * region_rank), 1, 10)                   # smaller regions have fewer staff

    base = np.array([float(typical.get(t, 25.0)) if pd.notna(typical.get(t, np.nan)) else 25.0 for t in request_type])
    handling = rng.lognormal(mean=np.log(base * (1 + 0.22 * region_rank)), sigma=0.55)   # right-skewed
    extreme = rng.random(n_rows) < 0.025
    handling[extreme] = handling[extreme] * rng.uniform(4, 10, size=int(extreme.sum()))   # extreme values
    peak = 1 + 0.9 * np.isin(hour, [9, 10, 11, 12])
    wait = rng.gamma(shape=2.0, scale=6.0, size=n_rows) * peak * (5.0 / staff)
    satisfaction = np.clip(np.round(5 - wait / 15 - handling / 120 + rng.normal(0, 0.6, n_rows)), 1, 5)

    truth = pd.DataFrame({"record_id": [f"R{i:05d}" for i in range(1, n_rows + 1)], "region": region,
                          "request_type": request_type, "channel": channel, "hour": hour.astype(int),
                          "staff_on_duty": staff.astype(float), "handling_minutes": np.round(handling, 1),
                          "wait_minutes": np.round(wait, 1), "satisfaction": satisfaction.astype(float)})
    messy = truth.copy()
    minority = regions[-1]
    gap_handling = 0.06 + 0.18 * (request_type == types[0]) + 0.10 * (region == minority)   # depends on type and region
    gap_satisfaction = 0.10 + 0.25 * (wait > np.quantile(wait, 0.75))                         # unhappy people answer less
    messy.loc[rng.random(n_rows) < gap_handling, "handling_minutes"] = np.nan
    messy.loc[rng.random(n_rows) < 0.08, "wait_minutes"] = np.nan                           # random gaps
    messy.loc[rng.random(n_rows) < gap_satisfaction, "satisfaction"] = np.nan
    messy.loc[rng.random(n_rows) < 0.05, "staff_on_duty"] = np.nan
    messy.loc[rng.random(n_rows) < 0.04, "request_type"] = np.nan
    return truth, messy

def planted_problems():
    """What was put into the data on purpose, so the exploration can be checked against it."""
    return pd.DataFrame([
        {"problem": "Missing values with a pattern", "where": "handling_minutes", "how": "more gaps for one request type and in the smallest region"},
        {"problem": "Missing values that depend on another column", "where": "satisfaction", "how": "more gaps when waiting time was long"},
        {"problem": "Random missing values", "where": "wait_minutes, staff_on_duty, request_type", "how": "4% to 8% of rows, no pattern"},
        {"problem": "High dispersion and extreme values", "where": "handling_minutes", "how": "2.5% of rows multiplied by 4 to 10"},
        {"problem": "Skewed distributions", "where": "handling_minutes, wait_minutes", "how": "log-normal and gamma shapes with a long right tail"},
        {"problem": "Imbalance / sampling bias", "where": "region", "how": "regions sampled in a 9 : 3 : 1 ratio"},
    ])

truth, messy = make_messy_dataset(result["analytical"])
repeat_truth, repeat_messy = make_messy_dataset(result["analytical"])
pd.testing.assert_frame_equal(messy, repeat_messy)                     # seeded => identical
print("Rows:", len(messy), "| columns:", messy.shape[1], "| empty cells:", int(messy.isna().sum().sum()))
display(planted_problems()); display(messy.head(8))

### AI-2 — Explore: find the problems before touching anything
- Measure missing values, and whether the gaps follow a **pattern** (a gap rate that changes from group to group).
- Measure the shape of every numeric column: skew, coefficient of variation (spread relative to the mean) and the share of extreme values by the IQR rule.
- Measure class balance, then write the list of problems with the evidence and the planned treatment.

In [ ]:
def missing_report(df):
    report = pd.DataFrame({"column": df.columns, "missing": [int(df[c].isna().sum()) for c in df.columns]})
    report["missing_pct"] = 100.0 * report["missing"] / len(df)
    return report.loc[report["missing"] > 0].sort_values("missing_pct", ascending=False, kind="stable").reset_index(drop=True)

def missing_patterns(df, by_columns=("request_type", "region")):
    """Does the gap rate of a column change between groups? A large spread means the gaps follow a pattern."""
    rows = []
    for column in df.columns:
        if df[column].isna().sum() == 0:
            continue
        for by in by_columns:
            if by == column:
                continue
            rate = df[column].isna().groupby(df[by]).mean() * 100.0
            rows.append({"column": column, "grouped_by": by, "lowest_gap_pct": float(rate.min()),
                         "highest_gap_pct": float(rate.max()), "highest_in": str(rate.idxmax()),
                         "spread_pct": float(rate.max() - rate.min())})
        if column != "wait_minutes" and "wait_minutes" in df.columns:
            long_wait = df["wait_minutes"] > df["wait_minutes"].quantile(0.75)
            rate = df[column].isna().groupby(long_wait).mean() * 100.0
            rows.append({"column": column, "grouped_by": "long wait (top 25%)", "lowest_gap_pct": float(rate.min()),
                         "highest_gap_pct": float(rate.max()), "highest_in": "long wait" if rate.idxmax() else "normal wait",
                         "spread_pct": float(rate.max() - rate.min())})
    table = pd.DataFrame(rows)
    table["follows_pattern"] = table["spread_pct"] > 8.0
    return table.sort_values("spread_pct", ascending=False, kind="stable").reset_index(drop=True)

def outlier_share(series):
    values = series.dropna()
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    spread = q3 - q1
    return float(((values < q1 - 1.5 * spread) | (values > q3 + 1.5 * spread)).mean())

def numeric_profile(df, columns=NUMERIC):
    rows = []
    for column in columns:
        values = df[column].dropna()
        rows.append({"column": column, "mean": float(values.mean()), "median": float(values.median()),
                     "std": float(values.std()), "cv": float(values.std() / values.mean()),
                     "skew": float(values.skew()), "outlier_share": outlier_share(values)})
    return pd.DataFrame(rows)

def balance_report(df, column=GROUP):
    counts = df[column].value_counts().sort_values(ascending=False, kind="stable")
    report = pd.DataFrame({column: counts.index, "rows": counts.to_numpy()})
    report["share_pct"] = 100.0 * report["rows"] / report["rows"].sum()
    return report, float(counts.max() / counts.min())

def find_problems(missing, patterns, profile, imbalance_ratio):
    """Turn the measurements into a to-do list: problem, evidence, planned treatment."""
    rows = []
    for record in missing.to_dict("records"):
        with_pattern = patterns.loc[(patterns["column"] == record["column"]) & patterns["follows_pattern"]]
        evidence = f"{record['missing_pct']:.1f}% missing"
        if len(with_pattern):
            evidence += f"; gap rate changes by {with_pattern.iloc[0]['grouped_by']} ({with_pattern.iloc[0]['spread_pct']:.0f} points)"
        rows.append({"problem": "Missing values" + (" with a pattern" if len(with_pattern) else ""), "column": record["column"],
                     "evidence": evidence, "planned_treatment": "compare fill methods on hidden known values (AI-3)"})
    for record in profile.to_dict("records"):
        if record["cv"] > 0.8 or (record["outlier_share"] > OUTLIER_LIMIT and abs(record["skew"]) > SKEW_LIMIT):
            rows.append({"problem": "High dispersion", "column": record["column"],
                         "evidence": f"CV {record['cv']:.2f}; {100 * record['outlier_share']:.1f}% beyond the IQR fences",
                         "planned_treatment": "cap extreme values inside each request type (AI-4)"})
        if abs(record["skew"]) > SKEW_LIMIT:
            rows.append({"problem": "Skewed distribution", "column": record["column"],
                         "evidence": f"skew {record['skew']:.2f}; mean {record['mean']:.1f} vs median {record['median']:.1f}",
                         "planned_treatment": "median for filling (AI-3), power transform before modelling (AI-4)"})
    if imbalance_ratio > 1.5:
        rows.append({"problem": "Imbalance / sampling bias", "column": GROUP,
                     "evidence": f"largest group is {imbalance_ratio:.1f}x the smallest",
                     "planned_treatment": "generate synthetic rows for small groups with SMOTE-NC (AI-5)"})
    return pd.DataFrame(rows)

def make_eda_figures(df, missing, balance):
    figures = {}
    fig, axes = plt.subplots(2, 2, figsize=(12, 7.6))
    ax = axes[0, 0]
    bars = ax.barh(missing["column"][::-1], missing["missing_pct"][::-1], color="#F4AD53")
    ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
    ax.set(title="Missing values by column", xlabel="% of rows missing"); ax.set_xlim(0, missing["missing_pct"].max() * 1.25)
    ax = axes[0, 1]
    bars = ax.bar(balance[GROUP], balance["rows"], color=["#4F63D9", "#20B9B0", "#F4AD53", "#9DAABD", "#D9534F"][:len(balance)])
    ax.bar_label(bars, padding=3, fontsize=9); ax.set(title="Rows per region (class balance)", ylabel="Rows")
    for ax, column in ((axes[1, 0], "handling_minutes"), (axes[1, 1], "wait_minutes")):
        values = df[column].dropna()
        ax.hist(values, bins=60, color="#4F63D9", alpha=.85)
        ax.axvline(values.mean(), color="#D9534F", linewidth=2, label=f"mean {values.mean():.1f}")
        ax.axvline(values.median(), color="#23344B", linewidth=2, linestyle="--", label=f"median {values.median():.1f}")
        ax.set(title=f"{column}: skew {values.skew():.2f}", xlabel="minutes", ylabel="Rows"); ax.legend(frameon=False)
    for ax in axes.ravel():
        ax.grid(axis="both", alpha=.15); ax.set_axisbelow(True); ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("AI-2 | What the raw data looks like", fontsize=14, weight="bold", x=0.02, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=140, facecolor="white", metadata={"Software": "Matplotlib"})
    plt.show(); plt.close(fig); figures["eda_overview.png"] = buf.getvalue()

    corr = df[MODEL_NUMERIC].corr()
    fig, ax = plt.subplots(figsize=(6.6, 5.4))
    im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr)), corr.columns, rotation=35, ha="right"); ax.set_yticks(range(len(corr)), corr.columns)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9,
                    color="white" if abs(corr.iloc[i, j]) > 0.5 else "#23344B")
    ax.set_title("AI-2 | Correlation between numeric columns", weight="bold", loc="left", pad=12)
    fig.colorbar(im, ax=ax, fraction=.046); fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=140, facecolor="white", metadata={"Software": "Matplotlib"})
    plt.show(); plt.close(fig); figures["eda_correlation.png"] = buf.getvalue()
    return figures

missing = missing_report(messy)
patterns = missing_patterns(messy)
profile = numeric_profile(messy)
balance, imbalance_ratio = balance_report(messy)
problems = find_problems(missing, patterns, profile, imbalance_ratio)
eda_figures = make_eda_figures(messy, missing, balance)
print("Missing values:"); display(missing.round(2))
print("Do the gaps follow a pattern?"); display(patterns.round(1))
print("Shape of numeric columns:"); display(profile.round(3))
print(f"Class balance (largest / smallest = {imbalance_ratio:.1f}):"); display(balance.round(1))
print("Problems found and planned treatment:"); display(problems)

### AI-3 — Study the gaps and fill them with the method that measures best
- **Mean or median?** Decided per column from its shape: the median when the column is strongly skewed or has many extreme values, otherwise the mean.
- **Which method?** Four candidates are tested by hiding 15% of the values we *do* know and measuring how far each method lands from them (mean absolute error, 5 seeded rounds): overall mean, overall median, the pattern-based statistic inside each request type × region group, and K-nearest-neighbours.
- A `_was_missing` flag is kept for every filled column, because the fact that a value was missing can itself be a signal for a model.
- At the end only, the fills are compared with the generator's hidden truth as a second check.

In [ ]:
def choose_statistic(series):
    """Median for skewed or heavy-tailed columns, mean otherwise."""
    values = series.dropna()
    return "median" if abs(float(values.skew())) > SKEW_LIMIT or outlier_share(values) > OUTLIER_LIMIT else "mean"

def fill_categorical(df, column, by=GROUP):
    """Most frequent value inside the group; overall most frequent value as a fallback."""
    filled = df[column].copy()
    overall = df[column].mode().iloc[0]
    for group, part in df.groupby(by):
        mode = part[column].mode()
        value = mode.iloc[0] if len(mode) else overall
        filled.loc[part.index[part[column].isna()]] = value
    return filled.fillna(overall)

def knn_fill(df, column, neighbours=7):
    """Fill from the most similar rows, using all numeric columns on a common scale."""
    values = df[MODEL_NUMERIC].to_numpy(dtype=float)
    mean, std = np.nanmean(values, axis=0), np.nanstd(values, axis=0)
    std[std == 0] = 1.0
    filled = KNNImputer(n_neighbors=neighbours).fit_transform((values - mean) / std) * std + mean
    return pd.Series(filled[:, MODEL_NUMERIC.index(column)], index=df.index)

def candidate_fills(df, column, statistic):
    """One full column of proposed values per method."""
    overall = df[column].median() if statistic == "median" else df[column].mean()
    by_group = df.groupby(["request_type", GROUP])[column].transform(statistic)
    by_region = df.groupby(GROUP)[column].transform(statistic)
    return {"overall mean": pd.Series(df[column].mean(), index=df.index),
            "overall median": pd.Series(df[column].median(), index=df.index),
            f"pattern: {statistic} per request type and region": by_group.fillna(by_region).fillna(overall),
            "k-nearest neighbours": knn_fill(df, column)}

def evaluate_fill_methods(df, column, statistic, rounds=5, hide_share=0.15, seed=AI_SEED):
    """Hide known values, fill them with each method, and measure the mean absolute error."""
    known = df.index[df[column].notna()].to_numpy()
    errors = {}
    for r in range(rounds):
        rng = np.random.default_rng(seed + r)
        hidden = rng.choice(known, size=max(1, int(hide_share * len(known))), replace=False)
        trial = df.copy()
        trial.loc[hidden, column] = np.nan
        for method, values in candidate_fills(trial, column, statistic).items():
            error = float((values.loc[hidden] - df.loc[hidden, column]).abs().mean())
            errors.setdefault(method, []).append(error)
    return {method: float(np.mean(values)) for method, values in errors.items()}

def impute_dataset(messy, truth=None):
    """Fill every gap. Returns (filled table, decisions table)."""
    filled = messy.copy()
    filled["request_type_was_missing"] = filled["request_type"].isna()
    filled["request_type"] = fill_categorical(filled, "request_type")
    decisions = []
    for column in NUMERIC:
        if filled[column].isna().sum() == 0:
            continue
        gaps = filled[column].isna()
        statistic = choose_statistic(filled[column])
        errors = evaluate_fill_methods(filled, column, statistic)
        best = min(errors, key=errors.get)
        values = candidate_fills(filled, column, statistic)[best]
        row = {"column": column, "gaps_filled": int(gaps.sum()), "skew": float(filled[column].skew()),
               "outlier_share": outlier_share(filled[column]), "mean_or_median": statistic, "method_used": best}
        for method, error in errors.items():
            row["error: " + ("pattern statistic" if method.startswith("pattern") else method)] = error
        if truth is not None:
            row["error_vs_hidden_truth"] = float((values[gaps] - truth.loc[gaps, column]).abs().mean())
            row["overall_mean_vs_hidden_truth"] = float((filled[column].mean() - truth.loc[gaps, column]).abs().mean())
        decisions.append(row)
        filled[column + "_was_missing"] = gaps
        filled.loc[gaps, column] = values[gaps]
    filled["staff_on_duty"] = filled["staff_on_duty"].round().clip(lower=1)
    filled["satisfaction"] = filled["satisfaction"].round().clip(1, 5)
    return filled, pd.DataFrame(decisions)

filled, fill_decisions = impute_dataset(messy, truth)
assert int(filled[NUMERIC + CATEGORICAL].isna().sum().sum()) == 0, "gaps remain after filling"
repeat_filled, _ = impute_dataset(messy, truth)
pd.testing.assert_frame_equal(filled, repeat_filled)
print("How each column was filled, and the measured error of every method (lower is better):")
display(fill_decisions.round(3))

### AI-4 — Treat dispersion and skew
- **Extreme values are capped, not deleted.** The fences are the IQR limits inside each request type, so a value is judged against its own kind. Deleting rows would throw away the other columns of those requests.
- **Skew is reduced with a power transform (Yeo-Johnson)** that is fitted to the data. Distance-based models such as K-Means are distorted by long tails.
- Note: synthetic-data generators do not fix dispersion. They are the tool for under-represented groups, which is the next step.

In [ ]:
def cap_outliers(df, columns=("handling_minutes", "wait_minutes"), by="request_type", minimum_group=20):
    """Clip values to the IQR fences of their own group (overall fences for very small groups)."""
    capped = df.copy()
    counts = {}
    for column in columns:
        q1, q3 = capped[column].quantile(0.25), capped[column].quantile(0.75)
        overall = (q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1))
        changed = 0
        for group, part in capped.groupby(by):
            low, high = overall
            if len(part) >= minimum_group:
                g1, g3 = part[column].quantile(0.25), part[column].quantile(0.75)
                low, high = g1 - 1.5 * (g3 - g1), g3 + 1.5 * (g3 - g1)
            low = max(low, 0.0)
            changed += int(((part[column] < low) | (part[column] > high)).sum())
            capped.loc[part.index, column] = part[column].clip(lower=low, upper=high)
        counts[column] = changed
    return capped, counts

def power_transform(values):
    """Yeo-Johnson transform fitted to one column; returns the transformed values."""
    return PowerTransformer(method="yeo-johnson", standardize=False).fit_transform(
        np.asarray(values, dtype=float).reshape(-1, 1)).ravel()

def treat_dispersion(filled):
    """Cap extreme values, then report spread and skew before and after (after = capped and transformed)."""
    capped, counts = cap_outliers(filled)
    rows = []
    for column in ("handling_minutes", "wait_minutes"):
        before, after = filled[column], capped[column]
        transformed = pd.Series(power_transform(after))
        rows.append({"column": column, "values_capped": counts[column],
                     "max_before": float(before.max()), "max_after": float(after.max()),
                     "cv_before": float(before.std() / before.mean()), "cv_after": float(after.std() / after.mean()),
                     "outlier_share_before": outlier_share(before), "outlier_share_after": outlier_share(after),
                     "skew_before": float(before.skew()), "skew_after_capping": float(after.skew()),
                     "skew_after_transform": float(transformed.skew())})
    return capped, pd.DataFrame(rows)

prepared, dispersion_report = treat_dispersion(filled)
assert len(prepared) == len(messy), "capping must not remove rows"
print("Dispersion and skew, before and after:"); display(dispersion_report.round(3))

### AI-5 — Balance the under-represented region with synthetic data (SMOTE-NC)
- **SMOTE** creates a new row between a real minority row and one of its nearest minority neighbours, so new rows follow the pattern of the real ones instead of copying them. **SMOTE-NC** is the version that also handles categorical columns.
- Every synthetic row is flagged `is_synthetic`, and real rows are never changed.
- **Quality check:** inside each region, the synthetic rows are compared with the real rows (means, and the Kolmogorov–Smirnov distance, where 0 means identical distributions).

In [ ]:
SMOTE_FEATURES = ["handling_minutes", "wait_minutes", "staff_on_duty", "satisfaction", "hour", "request_type", "channel"]

def balance_with_smote(prepared, seed=AI_SEED):
    """Oversample the small regions with SMOTE-NC until every region has as many rows as the largest."""
    X = prepared[SMOTE_FEATURES].copy()
    y = prepared[GROUP]
    categorical = [SMOTE_FEATURES.index(c) for c in CATEGORICAL]
    neighbours = int(min(5, y.value_counts().min() - 1))
    sampler = SMOTENC(categorical_features=categorical, random_state=seed, k_neighbors=neighbours)
    X_new, y_new = sampler.fit_resample(X, y)
    balanced = pd.DataFrame(X_new, columns=SMOTE_FEATURES)
    balanced[GROUP] = np.asarray(y_new)
    balanced["is_synthetic"] = np.arange(len(balanced)) >= len(prepared)      # new rows are appended at the end
    new = balanced["is_synthetic"].to_numpy()
    for column in ("handling_minutes", "wait_minutes"):
        balanced[column] = balanced[column].astype(float)
        balanced.loc[new, column] = balanced.loc[new, column].round(1)        # only the generated rows are rounded
    balanced["staff_on_duty"] = balanced["staff_on_duty"].astype(float).round().clip(lower=1)
    balanced["satisfaction"] = balanced["satisfaction"].astype(float).round().clip(1, 5)
    balanced["hour"] = balanced["hour"].astype(float).round().astype(int)
    ids = list(prepared["record_id"]) + [f"S{i:05d}" for i in range(1, len(balanced) - len(prepared) + 1)]
    balanced.insert(0, "record_id", ids)
    return balanced

def balance_table(prepared, balanced):
    before = prepared[GROUP].value_counts()
    rows = []
    for group in sorted(before.index, key=lambda g: -before[g]):
        part = balanced.loc[balanced[GROUP] == group]
        rows.append({GROUP: group, "real_rows": int(before[group]), "synthetic_rows": int(part["is_synthetic"].sum()),
                     "rows_after": int(len(part)), "share_before_pct": 100.0 * before[group] / len(prepared),
                     "share_after_pct": 100.0 * len(part) / len(balanced)})
    return pd.DataFrame(rows)

def synthetic_quality(balanced):
    """Inside each oversampled region: do synthetic rows look like the real ones?"""
    rows = []
    for group, part in balanced.groupby(GROUP):
        real, synthetic = part.loc[~part["is_synthetic"]], part.loc[part["is_synthetic"]]
        if len(synthetic) == 0:
            continue
        for column in ("handling_minutes", "wait_minutes", "staff_on_duty", "satisfaction"):
            rows.append({GROUP: group, "column": column, "real_mean": float(real[column].mean()),
                         "synthetic_mean": float(synthetic[column].mean()),
                         "ks_distance": float(ks_2samp(real[column], synthetic[column]).statistic)})
    return pd.DataFrame(rows)

balanced = balance_with_smote(prepared)
balance_after = balance_table(prepared, balanced)
quality = synthetic_quality(balanced)
pd.testing.assert_frame_equal(balanced, balance_with_smote(prepared))                       # seeded => identical
pd.testing.assert_frame_equal(balanced.loc[~balanced["is_synthetic"], SMOTE_FEATURES].reset_index(drop=True).astype(prepared[SMOTE_FEATURES].dtypes.to_dict()),
                              prepared[SMOTE_FEATURES].reset_index(drop=True))             # real rows untouched
assert balance_after["rows_after"].nunique() == 1, "regions are not balanced"
print("Rows per region, before and after:"); display(balance_after.round(1))
print("Do synthetic rows resemble the real rows of the same region?"); display(quality.round(3))

### AI-6 — Unsupervised model: does the preparation matter?
- The same model (K-Means, with the number of clusters chosen by the silhouette score) is trained on three versions of the data:
  1. **Naive:** drop every incomplete row, scale, nothing else.
  2. **Prepared:** gaps filled, extreme values capped, skew reduced.
  3. **Prepared + balanced:** the same, plus the SMOTE-NC rows.
- Reported for each: how many rows survive, how well separated the clusters are (silhouette), how stable they are when the random seed changes (agreement between runs, 1 = identical), and how much of the data comes from the smallest region.
- The verdict is written from these numbers. Silhouette values are comparable only roughly, because the three versions do not contain exactly the same rows.

In [ ]:
def model_features(df, treat):
    """Numeric columns (optionally power-transformed) on a common scale, plus one-hot categories."""
    numeric = df[MODEL_NUMERIC].astype(float).copy()
    if treat:
        for column in ("handling_minutes", "wait_minutes"):
            numeric[column] = power_transform(numeric[column])
    scaled = StandardScaler().fit_transform(numeric.to_numpy())
    dummies = pd.get_dummies(df[CATEGORICAL].astype(str), dtype=float)
    dummies = dummies.reindex(columns=sorted(dummies.columns))
    return np.hstack([scaled, dummies.to_numpy()])

def fit_clusters(X, seed=AI_SEED, k_values=(2, 3, 4, 5, 6)):
    """K-Means for several k; keep the k with the best silhouette, and measure stability across seeds."""
    best = None
    for k in k_values:
        labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X)
        score = float(silhouette_score(X, labels))
        if best is None or score > best[0]:
            best = (score, int(k), labels)
    score, k, labels = best
    agreement = []
    for other in range(1, 6):
        again = KMeans(n_clusters=k, n_init=10, random_state=seed + 100 * other).fit_predict(X)
        agreement.append(float(adjusted_rand_score(labels, again)))
    return labels, k, score, float(np.mean(agreement))

def compare_preparations(messy, prepared, balanced):
    """Train the same model on naive, prepared and prepared+balanced data."""
    complete = messy.dropna(subset=MODEL_NUMERIC + CATEGORICAL)
    versions = [("Naive: drop incomplete rows", complete, False),
                ("Prepared: filled, capped, de-skewed", prepared, True),
                ("Prepared + balanced (SMOTE-NC)", balanced, True)]
    rows, fitted = [], {}
    for name, data, treat in versions:
        X = model_features(data, treat)
        labels, k, score, stability = fit_clusters(X)
        shares = data[GROUP].value_counts(normalize=True)
        rows.append({"version": name, "rows_used": int(len(data)), "share_of_original_rows_pct": 100.0 * len(data) / len(messy),
                     "smallest_region_share_pct": 100.0 * float(shares.min()), "clusters": k,
                     "silhouette": score, "stability": stability})
        fitted[name] = (data, X, labels)
    return pd.DataFrame(rows), fitted

def cluster_points(data, X, labels, limit=1500, seed=AI_SEED):
    """Two-dimensional view (PCA) of the clustered rows, for plotting."""
    xy = PCA(n_components=2, random_state=seed).fit_transform(X)
    points = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "cluster": np.asarray(labels) + 1,
                           GROUP: data[GROUP].to_numpy(),
                           "is_synthetic": data["is_synthetic"].to_numpy() if "is_synthetic" in data else False})
    if len(points) > limit:
        keep = np.sort(np.random.default_rng(seed).choice(len(points), size=limit, replace=False))
        points = points.iloc[keep]
    return points.reset_index(drop=True)

def cluster_profile(data, labels):
    """What each cluster looks like, in the original units."""
    table = data.assign(cluster=np.asarray(labels) + 1)
    profile = table.groupby("cluster").agg(rows=("cluster", "size"), handling_minutes=("handling_minutes", "median"),
                                           wait_minutes=("wait_minutes", "median"), staff_on_duty=("staff_on_duty", "mean"),
                                           satisfaction=("satisfaction", "mean"), hour=("hour", "median")).reset_index()
    for group in sorted(table[GROUP].unique()):
        profile["share_" + str(group).lower()] = [float((table.loc[table["cluster"] == c, GROUP] == group).mean())
                                                  for c in profile["cluster"]]
    return profile

def make_prep_figures(messy, prepared, balance_after, points):
    figures = {}
    fig, axes = plt.subplots(2, 2, figsize=(12, 7.4))
    for row, column in enumerate(("handling_minutes", "wait_minutes")):
        before, after = messy[column].dropna(), pd.Series(power_transform(prepared[column]))
        axes[row, 0].hist(before, bins=60, color="#9DAABD"); axes[row, 0].set(title=f"{column} before: skew {before.skew():.2f}", ylabel="Rows")
        axes[row, 1].hist(after, bins=60, color="#20B9B0"); axes[row, 1].set(title=f"{column} after capping + transform: skew {after.skew():.2f}")
    for ax in axes.ravel():
        ax.grid(alpha=.15); ax.set_axisbelow(True); ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("AI-4 | Distributions before and after treatment", fontsize=14, weight="bold", x=0.02, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=140, facecolor="white", metadata={"Software": "Matplotlib"})
    plt.show(); plt.close(fig); figures["distributions_before_after.png"] = buf.getvalue()

    fig, ax = plt.subplots(figsize=(8.6, 4.6))
    x = np.arange(len(balance_after))
    real = ax.bar(x, balance_after["real_rows"], 0.6, color="#4F63D9", label="Real rows")
    synth = ax.bar(x, balance_after["synthetic_rows"], 0.6, bottom=balance_after["real_rows"], color="#F4AD53", label="Synthetic rows (SMOTE-NC)")
    ax.bar_label(real, label_type="center", color="white", fontsize=10)
    ax.bar_label(synth, labels=[str(v) if v else "" for v in balance_after["synthetic_rows"]], label_type="center", fontsize=10)
    ax.set_xticks(x, balance_after[GROUP]); ax.set(ylabel="Rows"); ax.legend(frameon=False)
    ax.set_title("AI-5 | Rows per region after balancing", weight="bold", loc="left", pad=12)
    ax.grid(axis="y", alpha=.15); ax.set_axisbelow(True); ax.spines[["top", "right"]].set_visible(False); fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=140, facecolor="white", metadata={"Software": "Matplotlib"})
    plt.show(); plt.close(fig); figures["balance_before_after.png"] = buf.getvalue()

    palette = ["#4F63D9", "#20B9B0", "#F4AD53", "#D9534F", "#9DAABD", "#23344B"]
    fig, ax = plt.subplots(figsize=(8.6, 5.6))
    for cluster in sorted(points["cluster"].unique()):
        part = points.loc[points["cluster"] == cluster]
        ax.scatter(part["x"], part["y"], s=12, alpha=.6, color=palette[(cluster - 1) % 6], label=f"Cluster {cluster}")
    ax.set(xlabel="PCA 1", ylabel="PCA 2"); ax.legend(frameon=False, markerscale=2)
    ax.set_title("AI-6 | Clusters on the prepared and balanced data (2-D view)", weight="bold", loc="left", pad=12)
    ax.spines[["top", "right"]].set_visible(False); fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=140, facecolor="white", metadata={"Software": "Matplotlib"})
    plt.show(); plt.close(fig); figures["clusters_pca.png"] = buf.getvalue()
    return figures

def prep_brief_facts(messy, missing, fill_decisions, dispersion_report, balance_after, comparison, imbalance_ratio):
    naive, ready, full = comparison.to_dict("records")
    handling = dispersion_report.set_index("column").loc["handling_minutes"]
    facts = {"rows": int(len(messy)), "empty_cells": int(messy.isna().sum().sum()),
             "columns_with_gaps": int(len(missing)), "worst_gap_pct": float(missing["missing_pct"].max()),
             "median_columns": int((fill_decisions["mean_or_median"] == "median").sum()),
             "fill_methods": sorted(fill_decisions["method_used"].str.split(":").str[0].unique()),
             "values_capped": int(dispersion_report["values_capped"].sum()),
             "handling_skew_before": float(handling["skew_before"]), "handling_skew_after": float(handling["skew_after_transform"]),
             "imbalance_before": float(imbalance_ratio), "synthetic_rows": int(balance_after["synthetic_rows"].sum()),
             "naive_rows_kept_pct": float(naive["share_of_original_rows_pct"]),
             "silhouette_naive": float(naive["silhouette"]), "silhouette_prepared": float(ready["silhouette"]),
             "silhouette_balanced": float(full["silhouette"]),
             "stability_naive": float(naive["stability"]), "stability_prepared": float(ready["stability"]),
             "stability_balanced": float(full["stability"]),
             "smallest_region_before_pct": float(ready["smallest_region_share_pct"]),
             "smallest_region_after_pct": float(full["smallest_region_share_pct"])}
    best = comparison.sort_values(["silhouette", "stability"], ascending=False, kind="stable").iloc[0]["version"]
    facts["best_separated_version"] = best
    if best.startswith("Naive"):
        verdict = ("On cluster separation alone the naive version scored highest, so the preparation is justified here by the rows "
                   "it saves and the group it restores, not by a better silhouette.")
    else:
        verdict = (f"The best-separated clusters came from the version “{best}”, so the preparation improved the model as well as "
                   "keeping more data.")
    brief = (
        f"This extension generated {facts['rows']} synthetic service requests with planted problems and prepared them for a model. "
        f"Exploration found {facts['empty_cells']} empty cells across {facts['columns_with_gaps']} columns (worst column "
        f"{facts['worst_gap_pct']:.0f}% missing), strongly skewed durations and a largest region {facts['imbalance_before']:.1f} times "
        f"the smallest. Each gap was filled by the method with the lowest measured error on hidden known values; "
        f"{facts['median_columns']} columns used the median because of their skew. {facts['values_capped']} extreme values were capped "
        f"inside their request type and a power transform cut the skew of handling time from {facts['handling_skew_before']:.2f} to "
        f"{facts['handling_skew_after']:.2f}. SMOTE-NC added {facts['synthetic_rows']} synthetic rows, lifting the smallest region from "
        f"{facts['smallest_region_before_pct']:.0f}% to {facts['smallest_region_after_pct']:.0f}% of the data. Dropping incomplete rows "
        f"instead would have kept only {facts['naive_rows_kept_pct']:.0f}% of the rows. K-Means silhouette was "
        f"{facts['silhouette_naive']:.2f} naive, {facts['silhouette_prepared']:.2f} prepared and {facts['silhouette_balanced']:.2f} "
        f"balanced, with stability {facts['stability_naive']:.2f}, {facts['stability_prepared']:.2f} and "
        f"{facts['stability_balanced']:.2f}. {verdict} "
        "Two limitations matter: the data is synthetic, and synthetic rows repeat the patterns of the few real rows they were built "
        "from, so they cannot add information that was never collected."
    )
    assert 120 <= len(brief.split()) <= 260, len(brief.split())
    return facts, brief

def write_prep_outputs(out_dir, messy, prepared, balanced, problems, fill_decisions, dispersion_report,
                       balance_after, quality, comparison, profile_table, brief, figures):
    """Write the AI project's files in stable order; returns the file names."""
    files = {"messy_dataset.csv": _csv_bytes(messy), "prepared_dataset.csv": _csv_bytes(prepared),
             "balanced_dataset.csv": _csv_bytes(balanced), "problems_found.csv": _csv_bytes(problems),
             "fill_decisions.csv": _csv_bytes(fill_decisions), "dispersion_report.csv": _csv_bytes(dispersion_report),
             "balance_report.csv": _csv_bytes(balance_after), "synthetic_quality.csv": _csv_bytes(quality),
             "model_comparison.csv": _csv_bytes(comparison), "cluster_profile.csv": _csv_bytes(profile_table),
             "ai_brief.txt": (brief + "\n").encode("utf-8"), **figures}
    destination = Path(out_dir)
    destination.mkdir(parents=True, exist_ok=True)
    for name in sorted(files):
        (destination / name).write_bytes(files[name])
    return sorted(files)

def refresh_results_zip(zip_path, ai_dir):
    """Replace the ai_output/ folder inside the results ZIP; every other entry is kept as it is."""
    with zipfile.ZipFile(zip_path) as old:
        kept = {name: old.read(name) for name in old.namelist() if not name.startswith("ai_output/")}
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name, payload in kept.items():
            archive.writestr(name, payload)
        for p in sorted(Path(ai_dir).iterdir()):
            archive.write(p, arcname=f"ai_output/{p.name}")

comparison, fitted = compare_preparations(messy, prepared, balanced)
repeat_comparison, _ = compare_preparations(messy, prepared, balanced)
pd.testing.assert_frame_equal(comparison, repeat_comparison)                 # seeded => identical
final_data, final_X, final_labels = fitted["Prepared + balanced (SMOTE-NC)"]
points = cluster_points(final_data, final_X, final_labels)
profile_table = cluster_profile(final_data, final_labels)
prep_figures = make_prep_figures(messy, prepared, balance_after, points)
ai_facts, AI_BRIEF = prep_brief_facts(messy, missing, fill_decisions, dispersion_report, balance_after, comparison, imbalance_ratio)
write_prep_outputs(AI_DIR, messy, prepared, balanced, problems, fill_decisions, dispersion_report, balance_after,
                   quality, comparison, profile_table, AI_BRIEF, {**eda_figures, **prep_figures})

for name, digest in result["manifest"]["files_sha256"].items():            # mandatory outputs untouched
    assert hashlib.sha256((OUTPUT_DIR / name).read_bytes()).hexdigest() == digest, name
refresh_results_zip(results_zip, AI_DIR)

print("Same model, three versions of the data:"); display(comparison.round(3))
print("Clusters on the final data (original units):"); display(profile_table.round(2))
print("\nAI brief (", len(AI_BRIEF.split()), "words):\n", AI_BRIEF)
print("\nAI outputs:", sorted(p.name for p in AI_DIR.iterdir()))
print("PASS: seeded and repeatable; no gaps left; real rows untouched; mandatory outputs unchanged")